# 02 · Features derivadas y prueba de no-fuga

La OBT del PSet #2 ya trae, por cada uno de los 22 atributos SMART, el valor del día y 18 columnas de ventana
(media, conteos y *shift* a 7, 14 y 30 días). Este notebook agrega **dos features derivadas** y demuestra que
**ninguna usa información del futuro**:

| Feature | Dónde se calcula | Qué mide |
|---|---|---|
| `DAYS_SINCE_FIRST_SEEN` | SQL de extracción (`src/datos.py`) | Edad observada del disco: días desde su primera observación |
| `N_CRITICAL_DEGRADING_30D` | `FeaturesDerivadas`, dentro del `Pipeline` (`src/pipeline.py`) | Cuántos atributos críticos de desgaste/errores empeoraron en los últimos 30 días |

Además, el `Pipeline` aplica `log1p` con signo a todas las columnas numéricas para los modelos lineales y la MLP
(hallazgo 3 de la EDA: colas largas). No hay variables categóricas que codificar: `MODEL_CODE` es constante (solo MC1)
y `SSD_KEY` es un identificador, que nunca entra al modelo.

Igual que la EDA, todo usa **solo el período de entrenamiento/CV** (2018-01 → 2019-09).

In [ ]:
import os, sys, json, time
from pathlib import Path

import numpy as np
import pandas as pd
import polars as pl_
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, str(Path.cwd().parent))
from src import datos, pipeline as pl

pd.set_option("display.width", 200, "display.max_columns", 30, "display.float_format", "{:,.4f}".format)
sns.set_theme(style="whitegrid", context="notebook")
DATOS = Path(os.environ.get("DATOS_DIR", "/datos"))
SNAPSHOT, CACHE = DATOS / "obt_mc1", DATOS / "cache_R"
FIG = Path("../docs/figuras"); FIG.mkdir(parents=True, exist_ok=True)
SEMILLA = pl.SEMILLA
T0 = time.time()

manifiesto = datos.leer_manifiesto(SNAPSHOT)
eda = json.loads(Path("../docs/eda_decisiones.json").read_text())
auc_eda = pd.read_csv("../docs/eda_pr_auc_univariada.csv", index_col=0)
eda

## 1. Datos

Se lee la **caché de modelado** (variante R elegida en la EDA): una matriz `float32` en disco construida desde el
snapshot y verificada contra su SHA-256. Un rango de fechas es un corte contiguo de filas, así que no se copia nada.

In [ ]:
t = time.time()
train = datos.cargar_cache(CACHE, hasta=datos.FIN_ENTRENAMIENTO, sha_esperado=manifiesto["dataset_sha256"])
columnas = pl.columnas_modelo(train, eda["variante"])
pl.verificar_sin_fuga(columnas)  # ninguna columna de etiqueta ni identificador entra a X
print(f"{len(train):,} filas · {len(columnas)} features de la variante {eda['variante']} · {time.time()-t:.1f} s")
print("Columnas prohibidas (nunca son features):", sorted(pl.COLUMNAS_PROHIBIDAS))

PREP = pl.preparar_eventos(train["SSD_KEY"], train["OBSERVATION_DATE"], train["DAYS_TO_FAILURE"])
Y = train["TARGET_30D"].to_numpy()

def pr_auc(x, signo=1):
    x = np.asarray(x, dtype=np.float64)
    return pl.curva_evento(None, None, None, np.where(np.isnan(x), -np.inf, signo * x), prep=PREP).pr_auc

AZAR = eda["pr_auc_evento_azar"]
print(f"Referencias de la EDA: azar {AZAR} · mejor atributo ({eda['atributo_regla']}) {eda['pr_auc_evento_regla_univariada']}")

## 2. Feature 1 · `DAYS_SINCE_FIRST_SEEN` (edad observada del disco)

**Por qué.** La EDA mostró que el parque envejece y falla más (hallazgo 2: la edad mediana pasa de 81 a 563 días
mientras la prevalencia se multiplica por 7,6). Las fallas de SSD siguen una curva de "bañera": hay más fallas al
principio (mortalidad infantil) y al final de la vida útil. La edad le da al modelo esa posición en la curva.

**Cómo se calcula (sin fuga).** En el SQL de extracción, **antes** de filtrar las filas con etiqueta:

```sql
DATEDIFF('day',
         MIN("OBSERVATION_DATE") OVER (PARTITION BY "SSD_KEY" ORDER BY "OBSERVATION_DATE"
                                       ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW),
         "OBSERVATION_DATE")
```

El marco de la ventana termina en la fila actual: solo ve el pasado. Se calcula antes del filtro de etiquetas para
que la edad cuente desde la primera observación real del disco, aunque esa fila no tenga etiqueta.

In [ ]:
print(datos.sql_extraccion("<OBT>", ["R_5"]))

In [ ]:
edad = train[datos.FEATURE_EDAD].to_numpy()
cortes = np.unique(np.nanquantile(edad, np.linspace(0, 1, 11)))
tramo = pd.cut(edad, cortes, include_lowest=True)
por_edad = (pd.DataFrame({"tramo": tramo, "y": Y})
            .groupby("tramo", observed=True)["y"].agg(filas="size", positivas="sum", prevalencia="mean"))
por_edad["lift"] = por_edad["prevalencia"] / Y.mean()

auc_edad = pr_auc(edad)
print(f"PR-AUC por evento de la edad: {auc_edad:.4f} = {auc_edad / AZAR:.1f}× el azar")

fig, ax = plt.subplots(figsize=(11, 3.6))
ax.bar(por_edad.index.astype(str), por_edad["lift"], color="#4C72B0")
ax.axhline(1, color="k", lw=1, ls="--")
ax.set_ylabel("lift de prevalencia"); ax.set_xlabel("edad observada (días), deciles")
ax.set_title("Riesgo de falla a 30 días por edad del disco")
ax.tick_params(axis="x", rotation=30)
fig.tight_layout(); fig.savefig(FIG / "02_lift_por_edad.png", dpi=150); plt.show()
por_edad

### 2.1 Prueba de no-fuga de la edad

Si la edad solo usa el pasado, entonces `fecha − edad` es **la misma fecha para todas las filas de un disco** (su
primera observación), y esa fecha es **anterior o igual** a su primera fila con etiqueta. Si dependiera del futuro
(por ejemplo, de la última fecha o de la fecha de falla), ese valor cambiaría entre filas o caería después.

In [ ]:
codigos = pl.codigos_disco(train["SSD_KEY"])
fecha_dias = pl.dias_desde_epoca(train["OBSERVATION_DATE"])
origen = fecha_dias - edad.astype(np.int64)
g = pd.DataFrame({"disco": codigos, "origen": origen, "fecha": fecha_dias}).groupby("disco")
resumen = g.agg(origenes=("origen", "nunique"), origen=("origen", "first"), primera_fila=("fecha", "min"))
n_constante = int((resumen["origenes"] == 1).sum())
n_pasado = int((resumen["origen"] <= resumen["primera_fila"]).sum())
print(f"Discos: {len(resumen):,}")
print(f"  'fecha − edad' constante en todo el disco: {n_constante:,} ({n_constante/len(resumen):.4%})")
print(f"  origen ≤ primera fila con etiqueta:       {n_pasado:,} ({n_pasado/len(resumen):.4%})")
print(f"  discos cuyo origen es anterior a su primera fila etiquetada: "
      f"{int((resumen['origen'] < resumen['primera_fila']).sum()):,} (su primera observación no tenía etiqueta)")
assert n_constante == len(resumen) and n_pasado == len(resumen), "La edad dependería del futuro"

## 3. Feature 2 · `N_CRITICAL_DEGRADING_30D` (atributos críticos empeorando)

**Por qué.** En la EDA, la señal más fuerte estaba en los contadores de errores y sectores (R_198, R_5, R_187,
R_197), y para varios la **tendencia** (`MEAN_SHIFT`) superaba al valor del día (hallazgo 5). Pero cada atributo por
separado es débil: un disco puede degradarse por cualquiera de ellos. Esta feature los resume en una sola cifra:

$$N\_CRITICAL\_DEGRADING\_30D(t) = \sum_{a \in \text{críticos}} \mathbb{1}\left[R\_a\_MEAN\_SHIFT\_30D(t) > 0\right]$$

`MEAN_SHIFT_30D` es la media de los días [t−29, t] menos la de [t−59, t−30]: mayor que 0 significa que el contador
creció en el último mes. Los atributos críticos son 5, 171, 172, 183, 184, 187, 197, 198 y 199 (reasignados,
errores de programa/borrado, errores no corregibles, timeouts, pendientes y no corregibles). Se excluyen 170 y 196
porque la EDA mostró que son copias exactas de 5, y 206 porque es copia de 171: se contarían dos veces.

Vive dentro del `Pipeline` (`FeaturesDerivadas`): se calcula igual en entrenamiento, validación, prueba e inferencia.

In [ ]:
cols_shift = [f"R_{a}_MEAN_SHIFT_30D" for a in pl.ATRIBUTOS_CRITICOS_DEFECTO]
deriv = pl.FeaturesDerivadas(salida="numpy").fit(train[cols_shift])
degr = deriv.transform(train[cols_shift])[:, -1]

por_valor = (pd.DataFrame({"n_degradando": degr.astype(int), "y": Y})
             .groupby("n_degradando")["y"].agg(filas="size", positivas="sum", prevalencia="mean"))
por_valor["pct_filas"] = por_valor["filas"] / len(train) * 100
por_valor["lift"] = por_valor["prevalencia"] / Y.mean()

auc_degr = pr_auc(degr)
mejor_shift = auc_eda.loc[[c for c in cols_shift if c in auc_eda.index], "pr_auc_evento"]
print(f"PR-AUC por evento de N_CRITICAL_DEGRADING_30D: {auc_degr:.4f} = {auc_degr / AZAR:.1f}× el azar")
print(f"Mejor MEAN_SHIFT_30D crítico por separado: {mejor_shift.idxmax()} = {mejor_shift.max():.4f}")

fig, ax = plt.subplots(figsize=(9, 3.6))
ax.bar(por_valor.index.astype(str), por_valor["lift"], color="#C44E52")
ax.axhline(1, color="k", lw=1, ls="--")
ax.set_xlabel("atributos críticos empeorando en 30 días"); ax.set_ylabel("lift de prevalencia")
ax.set_title("Riesgo de falla a 30 días según N_CRITICAL_DEGRADING_30D")
fig.tight_layout(); fig.savefig(FIG / "02_lift_degradacion.png", dpi=150); plt.show()
por_valor

### 3.1 Prueba de no-fuga de la degradación

Dos niveles:

1. **La feature es fila a fila.** Calcularla sobre los datos truncados en una fecha de corte da exactamente el mismo
   valor, para las filas anteriores al corte, que calcularla con todo el período: no mira otras filas.
2. **Sus insumos solo miran al pasado.** Para una muestra de discos se **recalculan** `R_a_MEAN_30D` y
   `R_a_MEAN_SHIFT_30D` desde el valor diario `R_a`, usando únicamente días ≤ t (ventanas [t−29, t] y [t−59, t−30]),
   y se comparan con las columnas de la OBT. Si el PSet #2 hubiera usado el futuro, no coincidirían.

In [ ]:
corte = pd.Timestamp("2019-03-31")
antes = train["OBSERVATION_DATE"].to_numpy() <= corte.to_datetime64()
truncado = deriv.transform(train.loc[antes, cols_shift])[:, -1]
assert np.array_equal(truncado, degr[antes]), "La feature cambió al truncar el futuro"
print(f"Truncando en {corte.date()}: {antes.sum():,} filas con el mismo valor (idénticas)")

In [ ]:
# Recalcular las ventanas desde el historial diario, solo con días <= t, para 500 discos.
rng = np.random.default_rng(SEMILLA)
discos = rng.choice(train["SSD_KEY"].cat.categories.to_numpy(), size=500, replace=False)
atribs = [5, 187, 198]
crudas = [f"R_{a}" for a in atribs]
obt_cols = [f"R_{a}_{s}" for a in atribs for s in ("MEAN_30D", "MEAN_SHIFT_30D")]
m = (pl_.from_pandas(train.loc[train["SSD_KEY"].isin(discos), ["SSD_KEY", "OBSERVATION_DATE"] + crudas + obt_cols]
                     .assign(SSD_KEY=lambda d: d["SSD_KEY"].astype(str)))
     .with_columns(pl_.col("OBSERVATION_DATE").cast(pl_.Date))
     .sort("SSD_KEY", "OBSERVATION_DATE"))
actual = m.rolling("OBSERVATION_DATE", period="30d", group_by="SSD_KEY").agg(
    [pl_.col(c).mean().alias(f"{c}_rec_actual") for c in crudas])
previa = m.rolling("OBSERVATION_DATE", period="30d", offset="-60d", group_by="SSD_KEY").agg(
    [pl_.col(c).mean().alias(f"{c}_rec_previa") for c in crudas])
rec = m.join(actual, on=["SSD_KEY", "OBSERVATION_DATE"]).join(previa, on=["SSD_KEY", "OBSERVATION_DATE"])

filas = []
for c in crudas:
    media = rec[f"{c}_rec_actual"].to_numpy()
    shift = media - rec[f"{c}_rec_previa"].to_numpy()
    for nombre, recalculado in ((f"{c}_MEAN_30D", media), (f"{c}_MEAN_SHIFT_30D", shift)):
        obt = rec[nombre].to_numpy().astype(np.float64)
        ambos = ~np.isnan(obt) & ~np.isnan(recalculado)
        iguales = np.isclose(obt[ambos], recalculado[ambos], rtol=1e-4, atol=1e-3)
        filas.append({"columna OBT": nombre, "filas comparadas": int(ambos.sum()),
                      "coinciden": int(iguales.sum()), "% coincide": iguales.mean() * 100})
verif = pd.DataFrame(filas).set_index("columna OBT")
print(f"{len(discos)} discos, {rec.height:,} filas")
verif

Las ventanas recalculadas con solo el pasado coinciden con la OBT, así que las columnas de ventana, y por lo tanto
`N_CRITICAL_DEGRADING_30D`, no usan información posterior a t. Si alguna fila no coincide, suele ser porque en su
ventana hay días sin etiqueta que la caché no contiene (por ejemplo, un tramo sin `TARGET_30D` en medio del
historial): en ese caso la OBT promedia **más días del pasado**, no del futuro.

## 4. Las features dentro del Pipeline

In [ ]:
from sklearn import set_config
set_config(display="text")
for familia in ("xgboost", "logistica"):
    modelo = pl.construir_modelo(familia)
    print(f"--- {familia}\n{modelo}\n")

# El mismo pipeline produce la feature en cualquier conjunto (entrenamiento, validación, prueba, inferencia)
muestra = train.iloc[:1000][columnas]
prep_arbol = pl.construir_preprocesador("xgboost").fit(muestra)
salida = prep_arbol.transform(muestra)
print("Árboles:", salida.shape, salida.dtype, "· última columna = N_CRITICAL_DEGRADING_30D")
prep_lineal = pl.construir_preprocesador("logistica").fit(muestra)
print("Lineales/MLP:", prep_lineal.transform(muestra).shape, "(log1p con signo + mediana + indicador de faltante + estándar)")

## 5. Resumen

In [ ]:
resumen = pd.DataFrame([
    {"feature": datos.FEATURE_EDAD, "dónde": "SQL de extracción", "PR-AUC por evento": auc_edad,
     "× azar": auc_edad / AZAR, "no-fuga": "fecha − edad constante por disco y ≤ primera fila"},
    {"feature": pl.FEATURE_DEGRADACION, "dónde": "FeaturesDerivadas (Pipeline)", "PR-AUC por evento": auc_degr,
     "× azar": auc_degr / AZAR, "no-fuga": "fila a fila + ventanas recalculadas con solo el pasado"},
]).set_index("feature")
Path("../docs/features_resumen.json").write_text(resumen.reset_index().to_json(orient="records", force_ascii=False, indent=2))
print(f"Tiempo total: {(time.time()-T0)/60:.1f} min")
resumen